# 09 存活分析：發病後，誰的預後比較差？

松柏護理之家 121 位感染住民中，19 人死亡。
主治醫師問：「哪些人的死亡風險比較高？能不能量化？」

**流程**：建立分析資料集 → KM 全體曲線 → 分組比較 → Log-rank 檢定 → Cox 迴歸 → HR 森林圖 → **PH 假設驗證**

> 💡 搭配 `09_survival.md` 閱讀，章節有白話解說、SVG 圖解、結果解讀指引；這個 notebook 是「看程式碼 + 看輸出」的實作版。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — 建立分析資料集

這個 cell 做三件事：
1. 讀入線列資料
2. 標記誰是「事件」（event=1 代表死亡）、誰是「設限」（event=0 代表仍存活）
3. 計算每個人的 `time_to_event`（天數）

> **小重點**：存活者的 `time_to_event` 用「最後發病日 + 14 天」當觀察截止，代表「我們至少看了這麼久都沒看到死亡」。這**不是** 0，也**不是**遺漏值。

In [ ]:
# --- Step 1: 建立存活分析資料集 ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 限定感染住民
cases = df[df["infected"] == 1].copy()

# 事件指標：1=死亡, 0=存活（右設限）
cases["event"] = (cases["outcome"] == "dead").astype(int)

# 存活時間
# 死亡者：time = death_date - symptom_onset_date
# 存活者：time = investigation_end - symptom_onset_date（設限）
investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

print(f"感染住民：{len(cases)}")
print(f"死亡：{cases['event'].sum()}")
print(f"存活（設限）：{(cases['event'] == 0).sum()}")
print(f"\n調查結束日：{investigation_end.date()}")
print(f"\n死亡者存活時間（天）：")
died = cases[cases["event"] == 1]
print(f"  mean = {died['time_to_event'].mean():.1f}")
print(f"  median = {died['time_to_event'].median():.1f}")
print(f"  range = {died['time_to_event'].min()} \u2013 {died['time_to_event'].max()} 天")

**結果解讀**：輸出應該顯示 121 位感染住民，19 位死亡、102 位設限。死亡者存活時間 range 告訴你「最快多少天就走了」和「最晚第幾天走」——對理解後面 KM 曲線的形狀很有幫助。

## Step 2 — Kaplan-Meier 全體存活曲線

KM 就是**看每天「還活著的比例」怎麼降**。每有人死 → 曲線下降一階；有人設限 → 曲線上標一個 tick。

> 搭配章節裡的 [KM 曲線拆解圖] 一起看，會很快掌握「階梯、tick、中位數、CI 帶」四要素。

In [ ]:
# --- Step 2: Kaplan-Meier 全體存活曲線 ---
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
kmf.fit(cases["time_to_event"], event_observed=cases["event"],
        label="全體感染住民")

fig, ax = plt.subplots(figsize=(8, 5))
kmf.plot_survival_function(ax=ax)
ax.set_title("Kaplan-Meier 存活曲線（全體感染住民）")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# 中位數存活時間
median_surv = kmf.median_survival_time_
print(f"中位存活時間：{median_surv}")
print("\u2192 如果中位數顯示 inf，代表超過 50% 的人在觀察期內存活（這是好消息）")

**怎麼讀這張曲線：**

1. **階梯下降** → 有住民在那天死亡
2. **tick 小豎線** → 仍存活的個案（設限）
3. **曲線穿過 y=0.5 的天數** → 中位存活時間；若永遠沒穿過 → `inf`（超過一半的人觀察期內沒死 ≈ **CFR 不到 50%**）
4. **陰影帶** = 95% 信賴區間，尾端越寬代表該時點樣本越少、不確定性越高

> 本案 CFR ≈ 15.7%，遠低於 50%，所以中位存活時間是 `inf`——這是**好消息**，不是 bug。

## Step 3 — 按嚴重度分組的存活曲線

把 cases 按 `clinical_severity` 分成 mild / moderate / severe 三組，疊三條 KM 曲線。

**看圖三視角**：
- **分離越早** → 因子效應越立竿見影
- **間距越大** → 效應越強
- **曲線交叉** → PH 假設可能被違反（Step 7 會驗證）

In [ ]:
# --- Step 3: 按嚴重度分組的存活曲線 ---
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

fig, ax = plt.subplots(figsize=(8, 5))

for sev in severity_levels:
    mask = cases["clinical_severity"] == sev
    sub = cases[mask]
    if len(sub) == 0:
        continue
    kmf_sev = KaplanMeierFitter()
    kmf_sev.fit(sub["time_to_event"], event_observed=sub["event"],
                label=f"{sev} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf_sev.plot_survival_function(ax=ax, color=colors[sev])

ax.set_title("存活曲線（按嚴重度分組）")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print("\u2192 觀察：severe 組的存活曲線是否明顯低於 mild/moderate？")
print("\u2192 曲線分離越早、距離越大 = 嚴重度對存活的影響越強")

**結果解讀**：

- 預期 `severe` 組的曲線最早、最快下降——因為重症病人短期死亡風險最高。
- 若 `severe` 曲線明顯低於 `mild`、`moderate`，代表嚴重度是重要的預後指標。
- 若觀察到 `severe` 和 `moderate` **交叉**，要特別記下來——這提示嚴重度的效應可能**隨時間變化**（不滿足 PH 假設）。

## Step 4 — Log-rank 檢定

Step 3 是**視覺判讀**，Step 4 是**統計推論**：

- **H₀**：兩組存活曲線形狀相同
- **H₁**：至少在某時點 hazard 不同
- **p < 0.05** → 拒絕 H₀，兩組差異顯著

⚠️ log-rank **只告訴你「有沒有差」**，**不告訴你差多少**——要量化效應（HR）得等 Step 5 的 Cox。

In [ ]:
# --- Step 4: Log-rank 檢定 ---
from lifelines.statistics import logrank_test

# 比較 severe vs non-severe
severe = cases[cases["clinical_severity"] == "severe"]
non_severe = cases[cases["clinical_severity"].isin(["mild", "moderate"])]

result = logrank_test(
    severe["time_to_event"], non_severe["time_to_event"],
    event_observed_A=severe["event"],
    event_observed_B=non_severe["event"],
)

print("=== Log-rank 檢定：severe vs non-severe ===")
print(f"  test statistic = {result.test_statistic:.3f}")
print(f"  p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("  \u2192 p < 0.05，兩組存活曲線有統計顯著差異")
else:
    print("  \u2192 p \u2265 0.05，無法拒絕兩組存活曲線相同的虛無假設")

# 也比較 COPD vs no COPD
copd_yes = cases[cases["comorbidity_copd"] == 1]
copd_no = cases[cases["comorbidity_copd"] == 0]

result_copd = logrank_test(
    copd_yes["time_to_event"], copd_no["time_to_event"],
    event_observed_A=copd_yes["event"],
    event_observed_B=copd_no["event"],
)

print(f"\n=== Log-rank 檢定：COPD vs no COPD ===")
print(f"  test statistic = {result_copd.test_statistic:.3f}")
print(f"  p-value = {result_copd.p_value:.4f}")

**結果解讀**：

- `test statistic` ≈ χ²(1) 分布，數值越大 → 兩組越不一樣
- `p_value` < 0.05 → 視為「統計顯著」；≥ 0.05 → 證據不足

> 想想：COPD 組人數可能不多（本案 ~28% 有 COPD）。如果 p 很大，可能是**真的沒差**，也可能是**樣本太少、力不足**——別把「不顯著」直接解讀成「無關」。

## Step 5 — Cox 比例風險迴歸

Cox 同時調整多個變項，告訴你**每個變項獨立的 HR**。

**讀 `print_summary()` 的口訣**：「看 `exp(coef)` 和它的 CI 就夠了」

| 欄名 | 意思 |
|------|------|
| `exp(coef)` | **HR**（1.5 = 風險快 1.5 倍） |
| `exp(coef) lower/upper 95%` | HR 的 95% CI；**跨過 1 → 不顯著** |
| `p` | p-value（< 0.05 顯著） |
| `Concordance` | 模型整體排序能力（0.5 隨機、>0.7 還可以、>0.8 好）|

In [ ]:
# --- Step 5: Cox 比例風險迴歸 ---
from lifelines import CoxPHFitter

# 建立 Cox 迴歸資料集
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "comorbidity_copd", "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "immunosuppressed",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

# 配適模型
cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox 比例風險迴歸結果 ===")
cph.print_summary()

# 簡潔的 HR 表格
print("\n=== Hazard Ratio 摘要 ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n\u2192 HR > 1 代表死亡風險較高（危險因子）")
print("\u2192 HR < 1 代表死亡風險較低（保護因子）")
print("\u2192 95% CI 包含 1 則不顯著")

**⚠️ 樣本數警訊（events per variable, EPV）**

- 本案只有 **19 個死亡事件**，卻放了 **7 個變項**（age, is_male, 4 個共病、immunosuppressed）
- 流行病學經驗法則：**每變項至少 10 個事件**（Peduzzi 1995）
- `19 / 7 ≈ 2.7`，遠低於 10 → **本章是教學示範**，實務上這樣的模型容易過度配適
- 這也解釋了為什麼很多變項「看似不顯著」——不是真的沒關係，而是樣本力不夠撐這麼多變項

**實務建議**：先做像 Ch06 的變項篩選（Modified Poisson crude RR），只留 1-2 個最關鍵的調整因子。

## Step 6 — HR 森林圖

`lifelines` 的 `cph.plot()` 畫的是 **log(HR)** —— 所以 x=0 那條垂直虛線代表 **HR = 1**。

**看三步驟**：
1. 點在 0 的哪一邊？→ 危險 vs 保護
2. 誤差線有沒有跨過 0？→ 跨過就不顯著
3. 誤差線多長？→ 越長代表 CI 越寬、不確定性越高

In [ ]:
# --- Step 6: HR 森林圖 ---
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression \u2014 Hazard Ratio（log scale）")
plt.tight_layout()
plt.show()

print("\u2192 森林圖中的點 = log(HR)，誤差線 = 95% CI")
print("\u2192 點在虛線右邊 = HR > 1（危險因子）")
print("\u2192 誤差線跨越虛線 = 不顯著")

**結果解讀**：

- **點 + 橫線「完全在 0 的右邊」** → 該變項是統計顯著的危險因子
- **點 + 橫線「完全在 0 的左邊」** → 統計顯著的保護因子
- **橫線跨過 x=0** → 95% CI 跨 1 → 不顯著
- **橫線很長** → 事件數少 / 變異大，估計不穩——實務上要謹慎解讀

> 本案事件少（19 events），許多變項的橫線會相當長——這是資料量的限制，不是 Cox 的錯。

## Step 7 — PH 假設驗證（全新）

Cox 的 **比例風險（PH）假設**：兩組 hazard 比值在整段追蹤期間保持常數。
如果違反，Step 5 算出來的 HR 會是「平均效應」，掩蓋了隨時間變化的真相。

用 `cph.check_assumptions()` 一行驗證：

- 對每個變項做 **Schoenfeld residuals** 檢定
- 印出每個變項的 p-value + 建議
- `No violation detected` → ✓ 通過；某個變項 `p < 0.05` → ✗ 違反

> `show_plots=False` 讓它只印文字結論（避免 notebook 被大量子圖塞爆）。

In [ ]:
# --- Step 7: PH 假設驗證 ---
print("=== 檢查比例風險（PH）假設 ===\n")

# show_plots=False：只印文字結論，避免多餘子圖
results = cph.check_assumptions(cox_df, show_plots=False, advice=True)

print("\n\u2192 全部變項都沒偵測到違反 \u2192 Cox 結果可信")
print("\u2192 若偵測到違反：可以用 strata、time-varying 係數，或改用 AFT 模型")

**結果解讀**：

| 輸出 | 意思 | 下一步 |
|------|------|--------|
| `proportional_hazard_test ... No violation detected` | ✓ 通過檢定 | Cox 結果可信 |
| 某個變項顯示 `p < 0.05` | ✗ 該變項違反 PH | 考慮補救（見下） |

**違反 PH 時的補救（由簡入深）**：
1. **分層（strata）**：`cph.fit(..., strata=["違反的變項"])` —— 允許該變項的 baseline hazard 自由變化
2. **時變係數（time-varying coefficient）**：讓該變項的效應隨時間變化
3. **拆時間段**：例如分「前兩週」「後兩週」分別跑 Cox
4. **改用 AFT（Accelerated Failure Time）模型**：完全繞開 PH 假設

⚠️ **事件少時的注意**：本案只有 19 events，`check_assumptions()` 檢定力不高——即使沒偵測到違反，也不代表 PH 一定成立。**永遠搭配 Step 3 的分組 KM 視覺檢查**（曲線有沒有交叉）。

## 補充 — COPD 分組 Kaplan-Meier

額外示範：用 `comorbidity_copd` 分兩組畫 KM 曲線，呼應 Step 4 的 log-rank 結果。

In [ ]:
# --- 補充：COPD 分組 Kaplan-Meier ---
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("COPD", cases["comorbidity_copd"] == 1),
                     ("No COPD", cases["comorbidity_copd"] == 0)]:
    sub = cases[mask]
    kmf_sub = KaplanMeierFitter()
    kmf_sub.fit(sub["time_to_event"], event_observed=sub["event"],
                label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf_sub.plot_survival_function(ax=ax)

ax.set_title("存活曲線：COPD vs No COPD")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print(f"\nCOPD Log-rank p-value = {result_copd.p_value:.4f}")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| Step 1 建立資料集 | 計算 `time_to_event` 與 `event` 指標，正確處理設限 |
| Step 2 KM 曲線 | `KaplanMeierFitter` 全體存活曲線；讀懂階梯、tick、中位數、CI 帶 |
| Step 3 分組 KM | 分組比較：看**分離時點**、**間距**、**是否交叉** |
| Step 4 Log-rank | `logrank_test()` 比較兩組差異；H₀/H₁、p-value 解讀 |
| Step 5 Cox 迴歸 | `CoxPHFitter` 多因子分析；讀懂 `print_summary()` 每一欄 |
| Step 6 森林圖 | `cph.plot()` 視覺化 HR；三步驟看圖 |
| **Step 7 PH 診斷** | `cph.check_assumptions()` 驗證 Cox 結果可不可信 |

**結論**：存活分析比單純的致死率（CFR）更精確——它同時考慮「有沒有死亡」和「多快死亡」。
Cox 迴歸可以同時調整多個因子，找出獨立的預後危險因子。但別忘了驗證 PH 假設，也別忽略 events-per-variable 的警訊。

下一章（Ch10），我們嘗試用全部特徵訓練機器學習模型 → 預測感染與重症。